# 02 - LoRA on SAM 3 (team stack)
The LoRA adaptation arm on the native `sam3` package, the shared team stack
(`Sam3MedSeg`). Build the model, verify the injection targets, and verify the
trainable-param counts of `finetune_mode="lora"`.

History (in git, not the working tree): the SAM 1 prototype lives in this
notebook's earlier versions (147K/93.7M, 0.157%); the HF-transformers SAM 3
verification was notebook 03_Sam3_explore.ipynb, removed after the team
standardized on the native stack (see commit 6d38175).

**Kernel: select the `sam3-native` env** (the only one with the `sam3` package).

In [1]:
#autoreload, so edits to .py files show without restarting
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.insert(0, "../src")   #repo convention so `medseg` imports
print(sys.executable)          #sanity check: should be the sam3-native env

/Users/nikhilclaude/anaconda3/envs/sam3-native/bin/python


## 1. Unit tests: LoRALayer / LinearWithLoRA / LoRA_qkv slice isolation

In [3]:
!pytest ../tests/test_lora.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.13, pytest-9.1.1, pluggy-1.6.0 -- /Users/nikhilclaude/anaconda3/envs/sam3-native/bin/python3.12
cachedir: .pytest_cache
rootdir: /Users/nikhilclaude/Documents/Projects/DeepLearning/group_project
plugins: anyio-4.14.2
collected 5 items                                                              

../tests/test_lora.py::test_output_shape PASSED                          [ 20%]
../tests/test_lora.py::test_zero_at_init PASSED                          [ 40%]
../tests/test_lora.py::test_param_count PASSED                           [ 60%]
../tests/test_lora.py::test_linear_with_lora PASSED                      [ 80%]
../tests/test_lora.py::test_lora_qkv_slices PASSED                       [100%]

============================== 5 passed in 0.73s ===============================


## 2. Build the native SAM 3 image model
Loads the gated weights from the HF cache (`load_from_HF=True` default), builds
on CPU (`device='cpu'` default). Takes a minute or two. This is the exact call
inside `Sam3Backbone.__init__`.

In [4]:
from sam3.model_builder import build_sam3_image_model

model = build_sam3_image_model()
total = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__}: {total:,} total params")

/Users/nikhilclaude/Documents/Projects/DeepLearning/group_project/vendor/sam3-repo/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/nikhilclaude/Documents/Projects/DeepLearning/group_project/vendor/sam3-repo/sam3/model/sam3_video_inference.py:800: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.bfloat16)
/Users/nikhilclaude/Documents/Projects/DeepLearning/group_project/vendor/sam3-repo/sam3/model/sam3_video_inference.py:908: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.bfloat16)
/Users/nikhilclaude/anaconda3/envs/sam3-native/lib/python3.12/site-packages/timm/models/layers/__in

Sam3Image: 840,509,750 total params


## 3. LoRA injection targets: 32 fused-qkv attention layers
Native vitdet fuses Q,K,V into ONE `Linear(1024, 3072)` per block (unlike the
HF edition's separate q_proj/v_proj), so we wrap with `LoRA_qkv`, not
`LinearWithLoRA`. Expect 32 blocks, each
`Linear(in_features=1024, out_features=3072, bias=True)`.

In [5]:
blocks = model.backbone.vision_backbone.trunk.blocks
print(f"{len(blocks)} blocks")
print(blocks[0].attn.qkv)

32 blocks
Linear(in_features=1024, out_features=3072, bias=True)


## 4. The LoRA arm: `Sam3MedSeg(finetune_mode="lora")`
Freeze the 455M-param backbone, inject adapters into all 32 qkv layers, train
adapters + FPN seg head. Expected (verified 2026-07-20):

```
1,066,113 / 455,105,921 trainable  (0.2343%)
  = 524,288 adapters (32 x 2 x 2*4*1024)  +  541,825 FpnSegHead
```

Any other number means the injection state is wrong (e.g. head swept into the
freeze, or a double injection; see docs/lora.md).

In [6]:
from medseg.models.sam3_medseg import Sam3MedSeg

m = Sam3MedSeg(finetune_mode="lora")
trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
total = sum(p.numel() for p in m.parameters())
print(f"{trainable:,} / {total:,} trainable  ({100*trainable/total:.4f}%)")

1,066,113 / 455,105,921 trainable  (0.2343%)


## 5. Smoke training run
Runs `scripts/train_lora.py`: tiny BUSI subset (8 images, 2 epochs, batch 1)
through the LoRA arm, using the shared Trainer (AdamW, BCE+Dice, Dice/NSD val).
Config: `configs/experiment/sam3_lora_busi.yaml`.

Notes:
- The `cd ..` matters: the script resolves `src/` and `configs/` relative to
  the repo root, and a notebook's working directory is `notebooks/`.
- First run downloads BUSI from Zenodo into `~/.medsegbench`.
- Success = train_loss drops between epochs and `best.pt` appears in
  `checkpoints/sam3_lora_busi_test/`. Slow on CPU; MPS is untested territory,
  if it errors switch the script's device to "cpu" and rerun.

In [ ]:
!cd .. && python scripts/train_lora.py